In [2]:
from pathlib import Path
from maomao.hierarchical_structure.define_hierarchy_and_structure import *

### Building a Hierarchically Structured Peptide Toxicity Resource

This notebook integrates the processed peptide toxicity datasets into a unified, sequence-level resource covering multiple toxicity endpoints. It consolidates direct annotations, resolves endpoint-specific ambiguous evidence, and applies positive-only hierarchical relationships between toxicity effects.

Ambiguous annotations always take precedence over positive evidence inferred from a child endpoint. Therefore, hierarchical propagation can add a positive annotation only when the parent endpoint is not already classified as ambiguous.

The final pivot contains one row per unique peptide sequence and one encoded column per toxicity endpoint:

| Code | Meaning |
|---:|---|
| `0` | Negative |
| `1` | Positive |
| `2` | Ambiguous |
| `3` | Unlabeled |
| `999` | No information |

The notebook also generates endpoint-level summaries and audit files to verify the input composition, hierarchical changes, and consistency of the resulting resource.

- Auxiliary variables

In [3]:
INPUT_ROOT = Path("../../processed_data/integrating_and_cleaning_data")
OUTPUT_ROOT = Path("../../processed_data/processed_data")

INCLUDE_FULL_CROSS_PRODUCT = True

MIN_LENGTH = 5
MAX_LENGTH = 70
CANONICAL_RESIDUES = "ACDEFGHIKLMNPQRSTVWY"

- Review input files: This cell shows which files exist.

In [4]:
cfg = Config(
    input_root=INPUT_ROOT,
    output_root=OUTPUT_ROOT,
    min_length=MIN_LENGTH,
    max_length=MAX_LENGTH,
    canonical_residues=CANONICAL_RESIDUES,
    include_full_cross_product=INCLUDE_FULL_CROSS_PRODUCT,
)

input_manifest = discover_input_files(cfg)
display(input_manifest)

,endpoint,input_endpoint_folder,declared_status,filename,path,exists
0,toxic,toxic,positive,positive.csv,../../processed_data/integrating_and_cleaning_...,True
1,toxic,toxic,negative,negative.csv,../../processed_data/integrating_and_cleaning_...,True
2,toxic,toxic,ambiguous,ambiguous_data.csv,../../processed_data/integrating_and_cleaning_...,True
3,toxic,toxic,unlabeled,only_unlabel.csv,../../processed_data/integrating_and_cleaning_...,False
4,cytotoxic,cytotoxic,positive,positive.csv,../../processed_data/integrating_and_cleaning_...,True
5,cytotoxic,cytotoxic,negative,negative.csv,../../processed_data/integrating_and_cleaning_...,True
6,cytotoxic,cytotoxic,ambiguous,ambiguous_data.csv,../../processed_data/integrating_and_cleaning_...,True
7,cytotoxic,cytotoxic,unlabeled,only_unlabel.csv,../../processed_data/integrating_and_cleaning_...,False
8,hemolytic,hemolytic,positive,positive.csv,../../processed_data/integrating_and_cleaning_...,True
9,hemolytic,hemolytic,negative,negative.csv,../../processed_data/integrating_and_cleaning_...,True


- Build the pivot file

In [5]:
results = build_all(cfg)

In [6]:
print("Unique sequences:", results["metadata"]["n_unique_sequences"])
print("Results saved in:", OUTPUT_ROOT.resolve())

Unique sequences: 71701
Results saved in: /home/nicole/Documentos/toxic_peptides_resource/processed_data/processed_data


- Summary and visualization of results

In [7]:
display(results["summary"]) # Summary data

status,toxicity_endpoint,positive,negative,ambiguous,unlabeled
0,cytolysis,341,2,0,0
1,cytotoxic,4697,19662,1094,0
2,embryotoxic,2,0,0,0
3,hemolytic,4867,19350,6697,5044
4,ichthyotoxic,5,0,0,0
5,neurotoxic,1481,1700,2,0
6,toxic,12049,37129,7769,0


In [8]:
print("\nPivote file:")
print("Shape:", results["wide"].shape)
display(results["wide"].head())


Pivote file:
Shape: (71701, 8)


,sequence,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic
0,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2,0,2,999,999,999,999
1,AAAAAAAAAGETS,999,0,0,999,999,999,999
2,AAAAAAAAAK,999,999,0,999,999,999,999
3,AAAAAAAIKMLMDLVNERIMALNKKAKK,0,0,0,999,999,999,999
4,AAAAARRRIRKQAHAHSK,0,0,0,999,999,999,999
